# 14 Case Study: Songbai Nursing Home Legionnaires' Disease Outbreak Investigation Report

Integrating skills from across the entire book, from raw data to a complete outbreak investigation report.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# -- CJK font setup (prevents CJK labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

plt.rcParams["font.size"] = 12
pd.set_option("display.max_columns", 40)

---
## 1. Background & Notification

In mid-January 2026, the health department received a notification from Songbai Nursing Home:
several residents had recently developed pneumonia symptoms, a suspected Legionnaires' disease outbreak.

- **Facility**: Songbai Nursing Home (280 residents)
- **Resident profile**: elderly residents aged 60–98, most with chronic conditions
- **Facilities**: 3 floors × 2 wings (A/B), with shower rooms and a hydrotherapy pool
- **Notification date**: January 2026

In [ ]:
# --- Step 1: Read data ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

date_cols = ["symptom_onset_date", "notification_date", "hospitalization_date",
             "death_date", "facility_admission_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Derived variables
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = df["clinical_severity"].isin(["severe"]).astype(int)
df["age_group"] = pd.cut(df["age"], bins=[59, 69, 79, 89, 100],
                         labels=["60-69", "70-79", "80-89", "90+"])

cases = df[df["infected"] == 1].copy()

print(f"Number of records: {len(df)} residents")
print(f"Columns: {df.shape[1]}")
print(f"Infected: {len(cases)}")
print(f"Date range: {cases['symptom_onset_date'].min().date()} ~ {cases['symptom_onset_date'].max().date()}")

---
## 2. Methods

- **Case definition**: a resident who developed pneumonia symptoms during the investigation period (`clinical_severity != 'not_ill'`)
- **Confirmed case**: a laboratory-confirmed Legionella infection (`lab_confirmed == 1`)
- **Data collection**: a retrospective investigation collecting demographic, exposure, and clinical data across 32 columns

---
## 3. Descriptive Epidemiology

In [ ]:
# --- 3a. Outbreak summary ---
n_total = len(df)
n_infected = int(df["infected"].sum())
n_confirmed = int(df["lab_confirmed"].sum())
n_hospitalized = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "Total residents": n_total,
    "Infected": n_infected,
    "Lab-confirmed": n_confirmed,
    "Hospitalized": n_hospitalized,
    "ICU": n_icu,
    "Deaths": n_deaths,
    "Attack rate (AR)": f"{n_infected / n_total:.1%}",
    "Case fatality rate (CFR)": f"{n_deaths / n_infected:.1%}",
}

print("=" * 40)
print("  Songbai Nursing Home Legionnaires' Disease Outbreak Summary")
print("=" * 40)
for k, v in summary.items():
    print(f"  {k}: {v}")
print("=" * 40)

In [ ]:
# --- 3b. Epidemic Curve ---
import matplotlib.dates as mdates

daily = cases.groupby("symptom_onset_date").size()
# Fill in the full date range (including 3 background days before the outbreak)
full_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
    freq="D",
)
daily = daily.reindex(full_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0,
       color="#2c7fb8", edgecolor="white", linewidth=0.5)
ax.set_title("Epidemic Curve of Legionnaires' Disease at Songbai Nursing Home, by Date of Onset, January 2026",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45)

ax.set_xlim(daily.index.min() - pd.Timedelta(hours=12),
            daily.index.max() + pd.Timedelta(hours=12))
ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Annotate the epidemic peak
peak_date = daily.idxmax()
ax.annotate(f"Peak: {peak_date.strftime('%m/%d')}\n({daily.max()} cases)",
            xy=(peak_date, daily.max()), xytext=(15, 5),
            textcoords="offset points", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="red"))

ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"→ The epidemic curve shows a common-source pattern")
print(f"→ Peak of onset: {peak_date.date()}, {daily.max()} cases total")

In [ ]:
# --- 3c. Person and place distribution ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Age distribution
for label, grp in df.groupby("infected"):
    tag = "Infected" if label == 1 else "Not infected"
    axes[0].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].set_title("Age distribution")
axes[0].legend()

# Sex x infection
sex_ct = df.groupby("sex")["infected"].agg(["sum", "count"])
sex_ct["ar"] = sex_ct["sum"] / sex_ct["count"] * 100
axes[1].bar(sex_ct.index, sex_ct["ar"], color=["#4C72B0", "#DD8452"])
axes[1].set_ylabel("Attack Rate (%)")
axes[1].set_title("Attack Rate by Sex")
for i, (idx, row) in enumerate(sex_ct.iterrows()):
    axes[1].text(i, row["ar"] + 1, f"{row['ar']:.1f}%", ha="center")

# Attack rate by floor and wing
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].set_ylabel("Attack Rate (%)")
axes[2].set_title("Attack Rate by Floor and Wing")
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"→ 2F-A ({zone[zone['label']=='2F-A']['ar'].values[0]:.1f}%) and 3F-B ({zone[zone['label']=='3F-B']['ar'].values[0]:.1f}%) had the highest attack rates")

---
## 4. Analytic Epidemiology

In [ ]:
# --- 4a. Shower exposure 2x2 table ---
ct = pd.crosstab(df["shower_use"], df["infected"], margins=True)
ct.index = ["No shower", "Shower", "Total"]
ct.columns = ["Not infected", "Infected", "Total"]
print("=== Shower use x infection status ===")
print(ct)

# Risk ratio (RR)
a, b = 76, 148  # shower & infected, total in shower group
c, d = 45, 132  # no shower & infected, total in no-shower group
a = int(df[(df["shower_use"] == 1) & (df["infected"] == 1)].shape[0])
b = int(df[df["shower_use"] == 1].shape[0])
c = int(df[(df["shower_use"] == 0) & (df["infected"] == 1)].shape[0])
d = int(df[df["shower_use"] == 0].shape[0])

rr = (a / b) / (c / d)
print(f"\nShower group attack rate: {a}/{b} = {a/b:.1%}")
print(f"No-shower group attack rate: {c}/{d} = {c/d:.1%}")
print(f"Risk ratio (RR): {rr:.2f}")

# Chi-square test
chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df["shower_use"], df["infected"]))
print(f"Chi-square test: χ² = {chi2:.2f}, p = {p:.4f}")
print(f"\n→ Shower use is significantly associated with infection (p < 0.05)" if p < 0.05 else "")

In [ ]:
# --- 4b. Stratified analysis: is functional_status a confounder? ---
print("=== Shower attack rate stratified by functional_status ===")
rows = []
for fs, grp in df.groupby("functional_status"):
    for su in [1, 0]:
        sub = grp[grp["shower_use"] == su]
        n = len(sub)
        cases_n = int(sub["infected"].sum())
        ar = cases_n / n * 100 if n > 0 else 0
        rows.append({"functional_status": fs, "shower_use": su,
                     "n": n, "cases": cases_n, "ar": f"{ar:.1f}%"})

strat_df = pd.DataFrame(rows)
print(strat_df.to_string(index=False))

print("\n→ Bedridden residents rarely shower and have lower infection rates")
print("→ functional_status is a confounder: it affects both showering ability and exposure opportunity")
print("→ Multivariable adjustment with logistic regression is needed")

In [ ]:
# --- 4c. Logistic regression: adjusted OR ---
import statsmodels.api as sm

model_df = df[["infected", "shower_use", "age", "sex",
               "comorbidity_chf", "comorbidity_dm", "comorbidity_copd",
               "immunosuppressed", "hydrotherapy_use"]].copy()
model_df["sex_male"] = (model_df["sex"] == "M").astype(int)
model_df = model_df.drop(columns=["sex"])

X = model_df.drop(columns=["infected"])
X = sm.add_constant(X)
y = model_df["infected"]

logit = sm.Logit(y, X).fit(disp=0)

# OR table
or_df = pd.DataFrame({
    "OR": np.exp(logit.params),
    "95% CI lower": np.exp(logit.conf_int()[0]),
    "95% CI upper": np.exp(logit.conf_int()[1]),
    "p-value": logit.pvalues,
}).drop(index="const")

or_df["OR (95% CI)"] = or_df.apply(
    lambda r: f"{r['OR']:.2f} ({r['95% CI lower']:.2f}–{r['95% CI upper']:.2f})", axis=1
)

print("=== Multivariable logistic regression results ===")
print(or_df[["OR (95% CI)", "p-value"]].to_string())
print("\n→ The adjusted OR for shower use (and its significance) after adjusting for age, sex, and comorbidities")

---
## 5. Time & Space Analysis

In [ ]:
# --- 5a. Comparison of epidemic curves across floor-wing zones ---
cases["zone"] = cases["floor"].astype(str) + "F-" + cases["wing"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
axes = axes.flatten()

for i, (zone_name, grp) in enumerate(cases.groupby("zone")):
    daily_z = grp.groupby("symptom_onset_date").size()
    daily_z = daily_z.reindex(full_range, fill_value=0)
    axes[i].bar(daily_z.index, daily_z.values, width=1.0,
                color="#2c7fb8", edgecolor="white", linewidth=0.5)
    n_cases = len(grp)
    n_total_zone = len(df[(df["floor"].astype(str) + "F-" + df["wing"]) == zone_name])
    ar = n_cases / n_total_zone * 100
    axes[i].set_title(f"{zone_name}  (AR={ar:.0f}%, n={n_cases})")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(False)
    axes[i].spines["top"].set_visible(False)
    axes[i].spines["right"].set_visible(False)
    axes[i].yaxis.set_major_locator(plt.MaxNLocator(integer=True))

fig.suptitle("Epidemic Curves of Legionnaires' Disease by Zone at Songbai Nursing Home, by Date of Onset, January 2026",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("→ The 2F and 3F-B zones show the densest epidemic curves")
print("→ Onset times across zones are similar, supporting the common-source hypothesis")

In [ ]:
# --- 5b. Spatial heatmap: attack rate x case fatality rate ---
zone_stats = df.groupby(["floor", "wing"]).agg(
    n=("infected", "count"),
    cases=("infected", "sum"),
    deaths=("outcome", lambda x: (x == "dead").sum()),
).reset_index()
zone_stats["ar"] = zone_stats["cases"] / zone_stats["n"] * 100
zone_stats["cfr"] = np.where(
    zone_stats["cases"] > 0,
    zone_stats["deaths"] / zone_stats["cases"] * 100,
    0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for idx, (metric, title, cmap) in enumerate([
    ("ar", "Attack Rate (%)", "YlOrRd"),
    ("cfr", "Case Fatality Rate (%)", "YlOrRd"),
]):
    pivot = zone_stats.pivot(index="floor", columns="wing", values=metric)
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap=cmap, ax=axes[idx],
                cbar_kws={"label": "%"})
    axes[idx].set_title(title)
    axes[idx].set_ylabel("Floor")
    axes[idx].set_xlabel("Wing")

plt.suptitle("Spatial Distribution: Attack Rate and Case Fatality Rate", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Advanced Analysis

In [ ]:
# --- 6a. Kaplan-Meier survival curve ---
from lifelines import KaplanMeierFitter

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["event"] = (cases["outcome"] == "dead").astype(int)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days
cases = cases[cases["time_to_event"] > 0]

fig, ax = plt.subplots(figsize=(8, 5))
kmf = KaplanMeierFitter()

for sev in ["severe", "moderate", "mild"]:
    mask = cases["clinical_severity"] == sev
    if mask.sum() > 0:
        kmf.fit(cases.loc[mask, "time_to_event"],
                cases.loc[mask, "event"], label=sev)
        kmf.plot_survival_function(ax=ax)

ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier Survival Curves (Stratified by Severity)")
plt.tight_layout()
plt.show()

print("→ Severe cases have markedly lower survival than moderate and mild cases")

In [ ]:
# --- 6b. Machine learning: ranking risk-factor importance ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = ["floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
            "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat_cols),
    ("bin", "passthrough", bin_cols),
])

pipe = Pipeline([
    ("pre", preprocess),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
])

X = df[num_cols + cat_cols + bin_cols]
y = df["infected"]

auc_scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")
print(f"Random Forest 5-fold CV AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")

# Feature importance
pipe.fit(X, y)
X_transformed = pipe.named_steps["pre"].transform(X)
feat_names = pipe.named_steps["pre"].get_feature_names_out()

perm = permutation_importance(pipe.named_steps["rf"], X_transformed, y,
                               n_repeats=10, random_state=42)

imp_df = pd.DataFrame({
    "feature": feat_names,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=True).tail(8)

# Map machine-generated feature names to human-readable labels
label_map = {
    "num__age": "Age",
    "cat__sex_M": "Sex (Male)",
    "cat__smoking_history_former": "Smoking history (Former)",
    "cat__smoking_history_current": "Smoking history (Current)",
    "cat__functional_status_wheelchair": "Functional status (Wheelchair)",
    "cat__functional_status_ambulatory": "Functional status (Ambulatory)",
    "cat__wing_B": "Wing (B)",
    "bin__floor": "Floor",
    "bin__comorbidity_chf": "Comorbidity: CHF",
    "bin__comorbidity_dm": "Comorbidity: Diabetes",
    "bin__comorbidity_cancer": "Comorbidity: Cancer",
    "bin__comorbidity_copd": "Comorbidity: COPD",
    "bin__immunosuppressed": "Immunosuppressed",
    "bin__shower_use": "Shower use",
    "bin__hydrotherapy_use": "Hydrotherapy use",
}
imp_df["label"] = imp_df["feature"].map(label_map).fillna(imp_df["feature"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["label"], imp_df["importance"], color="steelblue")
ax.set_xlabel("Permutation Importance")
ax.set_title("Predicting infection: top 8 most important features")
plt.tight_layout()
plt.show()

---
## 7. Discussion

### Source Identification

Bringing together all of the analyses above:

| Evidence | Finding | Points to |
|------|------|------|
| Epidemic curve | Common-source pattern | Persistent environmental exposure |
| Spatial distribution | Attack rate > 50% in 2F and 3F-B | Plumbing system on specific floors |
| Exposure analysis | Shower RR > 1, still significant after adjustment | Shower water as the transmission route |
| Stratified analysis | Bedridden residents (no showering) have low infection rates | Rules out airborne spread as the main cause |

**Conclusion: the shower water supply system is the most likely source of infection.**

### Recommended Interventions

1. **Immediate**: shut down the 2F and 3F-B shower facilities
2. **Short-term**: heat-disinfect the entire building's water system (> 70°C)
3. **Medium-term**: replace aging pipes and install water-temperature controls
4. **Long-term**: establish a routine water-quality monitoring program

In [ ]:
# --- 8. Conclusion: final report summary table ---
report = pd.DataFrame([
    ["Event type", "Legionnaires' disease outbreak"],
    ["Facility", "Songbai Nursing Home"],
    ["Investigation period", f"{cases['symptom_onset_date'].min().date()} ~ {cases['symptom_onset_date'].max().date()}"],
    ["Total residents", f"{n_total}"],
    ["Infected", f"{n_infected} (AR {n_infected/n_total:.1%})"],
    ["Deaths", f"{n_deaths} (CFR {n_deaths/n_infected:.1%})"],
    ["Hospitalized", f"{n_hospitalized}"],
    ["ICU", f"{n_icu}"],
    ["Highest-risk zones", "2F-A (54.5%), 3F-B (57.4%)"],
    ["Main risk factor", "Shower use"],
    ["Presumed source", "Shower water supply system"],
    ["Recommended measures", "Suspend showers → heat disinfection → replace pipes → routine monitoring"],
], columns=["Item", "Details"])

print("=" * 50)
print("    Songbai Nursing Home Legionnaires' Disease Outbreak — Final Report")
print("=" * 50)
for _, row in report.iterrows():
    print(f"  {row['Item']:　<10}{row['Details']}")
print("=" * 50)
print("\n→ This report is generated automatically by Python; every analysis step is reproducible")
print("→ See this notebook for the complete code")